# Imbalanced Image Classification

MNIST ships perfectly balanced — roughly 6,000 examples of every digit — which is exactly what real
datasets are not. Medical image sets are mostly healthy scans; defect-detection sets are mostly
good parts. This notebook deliberately breaks MNIST's balance, trains a CNN on the skewed result,
then rebalances by oversampling and trains the same architecture again, so the cost of imbalance
and the effect of the fix can be read off the same chart.

## Learning objectives

- Construct a deliberately imbalanced image dataset from a balanced one.
- Explain why overall accuracy is misleading when class frequencies are skewed.
- Read per-class recall from a classification report to find the classes a model is neglecting.
- Rebalance a training set with `RandomOverSampler` from imbalanced-learn.
- Compare a model trained on imbalanced data against the same architecture trained on rebalanced
  data.

## Background

You should already be able to build and train a small CNN, and to read a confusion matrix and a
classification report — including why **recall** for a class is the row of the confusion matrix
that matters when that class is rare.

Unit 1 covered the tabular version of this whole problem: `U1-7_Imbalance-2_Resampling.ipynb`
introduced over- and under-sampling, and `U1-6_Classify-3_F1Score.ipynb` covered why accuracy
misleads under skew. Nothing about that argument changes for images — the only new wrinkle is
mechanical, since an oversampler expects 2D tabular input and our images are 4D.

## This notebook covers

1. Building an imbalanced training set, and rebalancing it by oversampling
2. A baseline CNN trained on the imbalanced data
3. The same architecture trained on the rebalanced data
4. Comparing the two by per-class recall
5. Review

**Prerequisites:** `U2-2_CNN-2_MNIST.ipynb` for the CNN architecture and the training helper;
`U1-7_Imbalance-2_Resampling.ipynb` for resampling strategy.

**Dataset:** MNIST handwritten digits, loaded via `tensorflow.keras.datasets.mnist`, then
subsampled into a deliberately skewed training set.

**References:** https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Data preparation

### 1.1 Load MNIST

We start from the balanced original, reshaped to add the channel axis Keras convolutions expect.

In [ ]:
from tensorflow.keras.datasets import mnist

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

### 1.2 Create an imbalanced subsample

To manufacture a realistic skew we draw ten random class proportions from a log-normal
distribution: sample $z_d \sim \mathcal{N}(0, 1)$ for each digit $d$, exponentiate, and normalize
so the proportions sum to one,

$$ p_d = \frac{e^{z_d}}{\sum_{k=0}^{9} e^{z_k}} $$

Exponentiating is what produces the skew — a modest spread in $z$ becomes a large spread in $p$, so
the most common digit typically ends up with many times more examples than the rarest. The counts
change every run, which is deliberate: the point is the *shape* of the problem, not one particular
draw.

Note that only the **training** set is made imbalanced. The test set stays balanced at 250 images
per class, so every digit gets an equal vote in the evaluation and per-class recall is directly
comparable across classes. Evaluating on an equally-skewed test set would hide the exact failure we
are trying to expose.

The two sampling helpers are defined here, at their point of use.

In [ ]:
# Draw a random, deliberately skewed number of samples for each digit.
def sample_imbalanced(data, labels, num_samples):
    num_samples_per_digit = np.exp(np.random.randn(10))
    num_samples_per_digit = num_samples_per_digit / num_samples_per_digit.sum()
    num_samples_per_digit = np.floor(num_samples_per_digit * num_samples).astype(int)
    sampled_data = []
    sampled_labels = []
    for digit in range(10):
        digit_indices = np.where(labels == digit)[0]
        sampled_indices = np.random.choice(digit_indices, num_samples_per_digit[digit], replace=False)
        sampled_data.append(data[sampled_indices])
        sampled_labels.append(labels[sampled_indices])
    return np.concatenate(sampled_data), np.concatenate(sampled_labels).flatten()

# Take an EQUAL number of samples per digit -- used to keep the test set balanced.
def sample(data, labels, num_samples_per_digit):
    sampled_data = []
    sampled_labels = []
    for digit in range(10):
        digit_indices = np.where(labels == digit)[0]
        sampled_indices = np.random.choice(digit_indices, num_samples_per_digit, replace=False)
        sampled_data.append(data[sampled_indices])
        sampled_labels.append(labels[sampled_indices])
    return np.concatenate(sampled_data), np.concatenate(sampled_labels).flatten()

# Define how many samples of each digit to include
train_samples = 10000

# Sample the training data
X_train, y_train = sample_imbalanced(X_train, y_train, train_samples)

test_samples_per_class = 250

# Sample the testing data
X_test, y_test = sample(X_test, y_test, test_samples_per_class)

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)
print("y_train counts: ", np.bincount(y_train))

### 1.3 Oversample to balance

**Random oversampling** duplicates examples from the minority classes until every class has as many
rows as the largest one. No new information is created — the rare digits are simply repeated — but
the loss now weights each class equally, because each contributes an equal number of terms to the
sum.

One mechanical wrinkle: `imblearn` expects tabular input of shape `(n_samples, n_features)`, and
our images are `(N, 28, 28, 1)`. So we flatten to `(N, 784)`, resample, and reshape back. The
helper below does exactly that round trip.

We build the balanced set now and train on it in section 3 — section 2 first establishes what the
imbalanced data alone produces.

In [ ]:
from imblearn.over_sampling import RandomOverSampler

def oversample_with_imblearn(X, y):
    # Get original shape
    N, H, W, C = X.shape

    # Flatten for imblearn (samples, features)
    X_flat = X.reshape(N, -1)

    # Create oversampler
    ros = RandomOverSampler()

    # Perform oversampling
    X_res, y_res = ros.fit_resample(X_flat, y)

    # Reshape back to image form
    X_res = X_res.reshape(-1, H, W, C)

    return X_res, y_res

In [ ]:
X_train_bal, y_train_bal = oversample_with_imblearn(X_train, y_train)

print("Before balancing:", np.bincount(y_train))
print("After balancing: ", np.bincount(y_train_bal))

## 2. A baseline on the imbalanced data

Now train the CNN on the **skewed** training set and see what it costs.

Watch two things in the output. Overall accuracy will look respectable, because the common digits
dominate the average and the model does well on them. But the per-class **recall** column of the
classification report will sag badly for the rarest digits — the model has seen so few of them that
predicting a more common class is usually the better bet. That gap between the headline number and
the per-class numbers is the entire problem with imbalance.

### 2.1 Build the model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

dropout_rate = 0.25

n_classes  = np.unique(y_train).shape[0]

# Create model -- trained on the IMBALANCED data in this section.
model_imb = Sequential([
    Input(shape=X_train.shape[1:]),

    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(dropout_rate),
    
    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(dropout_rate),

    Flatten(),
    
    Dense(n_classes, activation='softmax'),
])

# Define the optimizer with a custom learning rate
optimizer = Adam(learning_rate=0.01)

# Compile model
model_imb.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=10,          # Stop after 10 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

# Display model summary
model_imb.summary()

### 2.2 Augment, train, and evaluate

Light augmentation is applied to both runs so the comparison in section 3 isolates one variable —
class balance — rather than confounding it with a change in the training recipe.

Evaluation happens on the **balanced** test set, so the classification report's recall column reads
as a fair per-digit score.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the image generator with augmentation options
datagen = ImageDataGenerator(
    rotation_range=10,      # Rotate images randomly
    width_shift_range=0.1,  # Randomly shift the width of images
    height_shift_range=0.1, # Randomly shift the height of images
    zoom_range=0.2,         # Randomly zoom
)

In [ ]:
# Train on the IMBALANCED training set (X_train / y_train).
model_imb, history_imb = helpers.train_and_evaluate(
    model_imb,
    datagen.flow(X_train, y_train, batch_size=512, shuffle=True),  # augmented, still skewed
    None,                                                          # labels come from the generator
    X_test, y_test,                                                # balanced test set for evaluation
    epochs=20,
    callbacks=[early_stopping],
    val_data=datagen.flow(X_test, y_test),
)

## 3. The same architecture on the rebalanced data

Now the comparison. Identical architecture, identical training recipe, identical augmentation — the
only change is that the training set is the **oversampled, balanced** one built in section 1.3.

A fresh model is constructed rather than continuing to train `model_imb`, so the two runs start from
the same footing and the difference is attributable to the data alone.

In [ ]:
# Same architecture as section 2.1, rebuilt from scratch so training starts fresh.
model_bal = Sequential([
    Input(shape=X_train_bal.shape[1:]),

    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(dropout_rate),

    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(),
    Dropout(dropout_rate),

    Flatten(),

    Dense(n_classes, activation='softmax'),
])

model_bal.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping_bal = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train on the BALANCED training set (X_train_bal / y_train_bal) -- the only difference.
model_bal, history_bal = helpers.train_and_evaluate(
    model_bal,
    datagen.flow(X_train_bal, y_train_bal, batch_size=512, shuffle=True),
    None,
    X_test, y_test,
    epochs=20,
    callbacks=[early_stopping_bal],
    val_data=datagen.flow(X_test, y_test),
)

## 4. Side by side: what balancing bought

Two confusion matrices are hard to compare by eye, so we plot the number that actually moved:
**per-class recall**, digit by digit, for both models.

Recall for digit $d$ is the fraction of true $d$ images the model actually found,

$$ \text{recall}_d = \frac{\text{TP}_d}{\text{TP}_d + \text{FN}_d} $$

which is the diagonal entry of the confusion matrix divided by its row total. It is the right
metric here because it asks a per-class question and is completely unaffected by how common the
class is — exactly the blind spot in overall accuracy.

Read the two panels together: the digits with the shortest bars on the left are the ones whose
recall should climb the most on the right.

In [ ]:
from sklearn.metrics import recall_score

# How skewed the training set actually was, and how each model scored per digit.
train_counts = np.bincount(y_train, minlength=10)

y_pred_imb = model_imb.predict(X_test, verbose=0).argmax(axis=1)
y_pred_bal = model_bal.predict(X_test, verbose=0).argmax(axis=1)

rec_imb = recall_score(y_test, y_pred_imb, average=None, labels=range(10))
rec_bal = recall_score(y_test, y_pred_bal, average=None, labels=range(10))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: the imbalance we manufactured in section 1.2.
axes[0].bar(range(10), train_counts, color='#4c72b0')
axes[0].set_title('Training images per digit (imbalanced set)')
axes[0].set_xlabel('digit'); axes[0].set_ylabel('count'); axes[0].set_xticks(range(10))

# Right: per-class recall, imbalanced training vs balanced training.
width = 0.4
axes[1].bar(np.arange(10) - width/2, rec_imb, width, label='trained imbalanced', color='#4c72b0')
axes[1].bar(np.arange(10) + width/2, rec_bal, width, label='trained balanced',   color='#dd8452')
axes[1].set_title('Per-class recall on the balanced test set')
axes[1].set_xlabel('digit'); axes[1].set_ylabel('recall')
axes[1].set_ylim(0, 1); axes[1].set_xticks(range(10)); axes[1].legend()

plt.tight_layout(); plt.show()

rarest = train_counts.argmin()
print(f"Rarest digit: {rarest}  ({train_counts.min()} training images vs {train_counts.max()} for the most common)")
print(f"Overall accuracy    -- imbalanced: {(y_pred_imb == y_test).mean():.3f}   balanced: {(y_pred_bal == y_test).mean():.3f}")
print(f"Worst-class recall  -- imbalanced: {rec_imb.min():.3f}   balanced: {rec_bal.min():.3f}")
print(f"Recall on digit {rarest}   -- imbalanced: {rec_imb[rarest]:.3f}   balanced: {rec_bal[rarest]:.3f}")

## 5. Review

| | Section 2 — imbalanced | Section 3 — rebalanced |
|---|---|---|
| Training set | Skewed by a log-normal draw, ~10,000 images | Oversampled so every digit matches the largest class |
| Architecture | 4 conv layers → Flatten → softmax | Identical |
| Augmentation | Light rotation/shift/zoom | Identical |
| Test set | Balanced, 250 per digit | Identical |
| What moves | — | Per-class recall on the rare digits |

**Takeaways**

- **Accuracy hides imbalance by construction.** It is an average weighted by class frequency, so a
  model can ignore the rarest digit entirely and still post a good headline number. The rare class
  contributes too few terms to the average to be noticed.
- **Per-class recall is the metric that sees the problem.** It asks a separate question for each
  class and is unaffected by how common that class is, which is exactly why the section 4 chart
  shows a gap the confusion matrices alone made hard to read.
- **Oversampling rebalances the loss, not the information.** Duplicating rare images adds no new
  examples of what those digits look like — it only makes each class contribute equally to the
  gradient. Expect the rare-class recall to rise while overall accuracy moves very little, and
  possibly down slightly, since the common classes give up some of their advantage.
- **Never rebalance the test set.** The training set is a knob you control; the test set is the
  measurement. Skewing it too would hide precisely the failure you are trying to detect, which is
  why only `X_train` was made imbalanced here.
- **This is Unit 1's lesson unchanged.** `U1-7_Imbalance-2_Resampling.ipynb` made the same argument
  on tabular data. The only image-specific detail is the flatten-resample-reshape round trip needed
  to push 4D arrays through `imblearn`.

**Where to take it further:** class weights (`class_weight` in `model.fit`) achieve a similar
rebalancing without duplicating data, and are often preferable for large image sets where copying
rows is expensive — see `U1-7_Imbalance-4_ClassWeights.ipynb`.

**Next:** `U2-2_CNN-4_Cifar.ipynb` moves from gray-scale digits to color photographs.